# ADMM TEST

In [ ]:
import time
import struct
import numpy as np
import scipy.sparse as sp
from pynq import Overlay, allocate, MMIO

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================
# IMPORTANT: Update this path to your actual bitstream location
bitstream_path = '/home/xilinx/admm/admm.bit' 
print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print('Overlay loaded!')

CONTROL_ADDR   = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE     = 0x10000

print(f"Manually mapping Control   : {hex(CONTROL_ADDR)}")
print(f"Manually mapping Control_R : {hex(CONTROL_R_ADDR)}")

control_ip   = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

In [ ]:
import time
import struct
import numpy as np
import scipy.sparse as sp
from pynq import Overlay, allocate, MMIO

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================
# IMPORTANT: Update this path to your actual bitstream location
bitstream_path = '/home/xilinx/admm/admm.bit'
print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print('Overlay loaded!')

CONTROL_ADDR = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE = 0x10000

print(f"Manually mapping Control   : {hex(CONTROL_ADDR)}")
print(f"Manually mapping Control_R : {hex(CONTROL_R_ADDR)}")

control_ip = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

# =====================================================================
# 2. Problem Setup (Adaptive rho stress test)
#    Goal: force >50 ADMM iterations so the adaptive update triggers.
# =====================================================================
print("\nRunning ADMM accelerator adaptive-rho stress test...")

# ------------------------------
# Tunable knobs (start here)
# ------------------------------
SEED = 123
RUN_BOTH_ADAPTIVE_RHO = True

# Keep 15x15 unless you want a heavier test
num_rows = 15
num_cols = 15
PACK_SIZE = 16

# Kernel parameters
alpha = 1.0        # 1.6 is faster; 1.0 makes convergence slower/more stable for this stress test
sigma = 1e-6
admm_max_iterations = 2000
pcg_max_iterations = 200

# Stress-test design: objective wants x_obj, constraints are built around a different feasible x_feas.
# With a very small initial rho, the solver should need many iterations until adaptive rho kicks in at 50.
A_COND_LOG10 = 4.0          # A condition number ~ 1e4
A_MIN_ABS = 1e-6            # keep A entries non-zero in CSC
P_LOG10_MIN = -2.0          # P diagonal spans [1e-2, 1e2]
P_LOG10_MAX = 2.0
SLACK_MIN = 1e-5            # tight bounds => harder
SLACK_MAX = 1e-4
ONE_SIDED_PROB = 0.7        # many constraints active at solution
EQ_PROB = 0.0               # avoid exact equalities (keeps r_dual nonzero)

# IMPORTANT: must be > ~1.1e-6 or kernel update_rho() will refuse to scale it
RHO_INIT = 1.5e-6

# OSQP / solver constants (match kernel expectations)
OSQP_INFTY = 1e20
OSQP_RHO_MAX = 1e6
OSQP_RHO_MIN = 1e-6

np.random.seed(SEED)

def rand_orthonormal(n):
    m = np.random.randn(n, n)
    q, _ = np.linalg.qr(m)
    return q.astype(np.float32)

# 1) Build dense, ill-conditioned A via SVD-like construction: A = U * diag(s) * V^T
k = min(num_rows, num_cols)
U = rand_orthonormal(num_rows)
V = rand_orthonormal(num_cols)
s = np.logspace(0.0, -A_COND_LOG10, k).astype(np.float32)
S = np.zeros((num_rows, num_cols), dtype=np.float32)
S[np.arange(k), np.arange(k)] = s
A_dense = (U @ S @ V.T).astype(np.float32)

# Prevent entries being *exactly* zero (so CSC nnz stays dense and deterministic)
mask = np.abs(A_dense) < A_MIN_ABS
if np.any(mask):
    A_dense[mask] = np.where(A_dense[mask] >= 0.0, A_MIN_ABS, -A_MIN_ABS).astype(np.float32)

A_sparse = sp.csc_matrix(A_dense, dtype=np.float32)
A_col_ptr = A_sparse.indptr.astype(np.int32)
A_row_idx = A_sparse.indices.astype(np.int32)
A_values = A_sparse.data.astype(np.float32)
A_nnz = int(A_sparse.nnz)

# Transpose A (CSC)
AT_sparse = A_sparse.transpose().tocsc()
AT_col_ptr = AT_sparse.indptr.astype(np.int32)
AT_row_idx = AT_sparse.indices.astype(np.int32)
AT_values = AT_sparse.data.astype(np.float32)

# 2) Diagonal P and conflicting objective
P_values = (10.0 ** np.random.uniform(P_LOG10_MIN, P_LOG10_MAX, num_cols)).astype(np.float32)
P_row_idx = np.arange(num_cols, dtype=np.int32)
P_col_ptr = np.arange(num_cols + 1, dtype=np.int32)
P_diag = P_values.copy()
P_nnz = int(num_cols)

x_obj = np.random.uniform(-2.0, 2.0, num_cols).astype(np.float32)
q = (-P_values * x_obj).astype(np.float32)  # unconstrained minimizer is x_obj
x_feas = (-x_obj).astype(np.float32)        # ensure constraints conflict with the objective
z_center = (A_sparse @ x_feas).astype(np.float32)

# 3) Tight bounds around z_center, and very small initial rho
l_in = np.empty(num_rows, dtype=np.float32)
u_in = np.empty(num_rows, dtype=np.float32)
slack = np.random.uniform(SLACK_MIN, SLACK_MAX, num_rows).astype(np.float32)

for i in range(num_rows):
    if np.random.rand() < EQ_PROB:
        l_in[i] = z_center[i]
        u_in[i] = z_center[i]
        continue

    s_i = slack[i]
    if np.random.rand() < ONE_SIDED_PROB:
        if np.random.rand() < 0.5:
            l_in[i] = z_center[i]
            u_in[i] = z_center[i] + s_i
        else:
            l_in[i] = z_center[i] - s_i
            u_in[i] = z_center[i]
    else:
        l_in[i] = z_center[i] - s_i
        u_in[i] = z_center[i] + s_i

rho_in = np.full(num_rows, RHO_INIT, dtype=np.float32)
rho_in = np.clip(rho_in, 1.2e-6, OSQP_RHO_MAX).astype(np.float32)

print(f"Problem Size: {num_rows}x{num_cols}")
print(f"A_nnz: {A_nnz} (avg nnz/col: {A_nnz / num_cols:.2f})")
print(f"P diag range: [{P_values.min():.2e}, {P_values.max():.2e}]")
print(f"Initial rho range: [{rho_in.min():.2e}, {rho_in.max():.2e}]")

def max_box_violation(z, l, u):
    return float(np.max(np.maximum(0.0, np.maximum(l - z, z - u))))

z_obj = (A_sparse @ x_obj).astype(np.float32)
viol_obj = max_box_violation(z_obj, l_in, u_in)
viol_feas = max_box_violation(z_center, l_in, u_in)
print(f"Max constraint violation (x_obj):  {viol_obj:.3e}")
print(f"Max constraint violation (x_feas): {viol_feas:.3e}  (should be 0)")

# =====================================================================
# Export Matrix and Vector data to reuse in C test
# =====================================================================
import os

output_dir = "/home/xilinx/admm/baremetal/data"
os.makedirs(output_dir, exist_ok=True)

x_obj.tofile(os.path.join(output_dir, "x_obj.bin"))
x_feas.tofile(os.path.join(output_dir, "x_feas.bin"))
P_values.astype(np.float32).tofile(os.path.join(output_dir, "P_values.bin"))
P_col_ptr.astype(np.int32).tofile(os.path.join(output_dir, "P_col_ptr.bin"))
A_values.astype(np.float32).tofile(os.path.join(output_dir, "A_values.bin"))
A_col_ptr.astype(np.int32).tofile(os.path.join(output_dir, "A_col_ptr.bin"))
A_row_idx.astype(np.int32).tofile(os.path.join(output_dir, "A_row_idx.bin"))
l_in.astype(np.float32).tofile(os.path.join(output_dir, "l_in.bin"))
u_in.astype(np.float32).tofile(os.path.join(output_dir, "u_in.bin"))
q.astype(np.float32).tofile(os.path.join(output_dir, "q.bin"))
rho_in.astype(np.float32).tofile(os.path.join(output_dir, "rho_in.bin"))

print(f"Matrices successfully dumped to: {output_dir}")

# =====================================================================
# 3. Buffer Allocation and Packing
# =====================================================================
def ceil_div(a, b):
    return (a + b - 1) // b

def pack_csc_to_words(row_idx, values, num_words):
    nnz = len(row_idx)
    row_words = allocate(shape=(num_words, PACK_SIZE), dtype=np.int32, cacheable=False)
    val_words = allocate(shape=(num_words, PACK_SIZE), dtype=np.float32, cacheable=False)
    row_words[:] = 0
    val_words[:] = 0.0
    row_words.reshape(-1)[:nnz] = row_idx
    val_words.reshape(-1)[:nnz] = values
    return row_words, val_words

print("Allocating and Packing Hardware CMA Buffers (Uncached)...")

A_words = int(ceil_div(A_nnz, PACK_SIZE))
AT_words = int(ceil_div(A_nnz, PACK_SIZE))
P_words = int(ceil_div(P_nnz, PACK_SIZE))

A_row_words, A_val_words = pack_csc_to_words(A_row_idx, A_values, A_words)
AT_row_words, AT_val_words = pack_csc_to_words(AT_row_idx, AT_values, AT_words)
P_row_words, P_val_words = pack_csc_to_words(P_row_idx, P_values, P_words)

# Pad pointers to ensure AXI bursts don't read out of bounds
PAD = 16

A_col_ptr_hw = allocate(shape=(num_cols + 1 + PAD,), dtype=np.int32, cacheable=False)
A_col_ptr_hw[: num_cols + 1] = A_col_ptr

AT_col_ptr_hw = allocate(shape=(num_rows + 1 + PAD,), dtype=np.int32, cacheable=False)
AT_col_ptr_hw[: num_rows + 1] = AT_col_ptr

P_col_ptr_hw = allocate(shape=(num_cols + 1 + PAD,), dtype=np.int32, cacheable=False)
P_col_ptr_hw[: num_cols + 1] = P_col_ptr

P_diag_hw = allocate(shape=(num_cols + PAD,), dtype=np.float32, cacheable=False)
P_diag_hw[:num_cols] = P_diag

l_in_hw = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False)
l_in_hw[:num_rows] = l_in

u_in_hw = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False)
u_in_hw[:num_rows] = u_in

q_in_hw = allocate(shape=(num_cols + PAD,), dtype=np.float32, cacheable=False)
q_in_hw[:num_cols] = q

rho_in_hw = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False)
rho_in_hw[:num_rows] = rho_in

x_out_hw = allocate(shape=(num_cols + PAD,), dtype=np.float32, cacheable=False)
y_out_hw = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False)
x_out_hw[:] = 0.0
y_out_hw[:] = 0.0

# =====================================================================
# 4. Hardware Execution
# =====================================================================
def write_64bit_address(ip, base_offset, address):
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

def float_to_uint(f_val):
    return struct.unpack('<I', struct.pack('<f', float(f_val)))[0]

def uint_to_float(u_val):
    return struct.unpack('<f', struct.pack('<I', int(u_val)))[0]

def objective_val(x_vec):
    # P is diagonal here
    return float(0.5 * np.sum(P_diag * x_vec * x_vec) + np.dot(q, x_vec))

def run_hw(adaptive_rho):
    print(f"\n=== HW Run (adaptive_rho={adaptive_rho}) ===")
    print("Configuring Registers...")

    # Write to Control Registers (0xA0000000)
    control_ip.write(0x10, num_rows)
    control_ip.write(0x18, num_cols)
    control_ip.write(0x20, A_nnz)
    control_ip.write(0x28, P_nnz)
    control_ip.write(0x30, float_to_uint(sigma))
    control_ip.write(0x38, float_to_uint(alpha))
    control_ip.write(0x40, admm_max_iterations)
    control_ip.write(0x48, pcg_max_iterations)
    control_ip.write(0x50, int(adaptive_rho))

    # Write to Control_R Registers (Addresses to 0xA0010000)
    write_64bit_address(control_r_ip, 0x10, A_row_words.device_address)
    write_64bit_address(control_r_ip, 0x1c, A_col_ptr_hw.device_address)
    write_64bit_address(control_r_ip, 0x28, A_val_words.device_address)

    write_64bit_address(control_r_ip, 0x34, AT_row_words.device_address)
    write_64bit_address(control_r_ip, 0x40, AT_col_ptr_hw.device_address)
    write_64bit_address(control_r_ip, 0x4c, AT_val_words.device_address)

    write_64bit_address(control_r_ip, 0x58, P_row_words.device_address)
    write_64bit_address(control_r_ip, 0x64, P_col_ptr_hw.device_address)
    write_64bit_address(control_r_ip, 0x70, P_val_words.device_address)

    write_64bit_address(control_r_ip, 0x7c, P_diag_hw.device_address)
    write_64bit_address(control_r_ip, 0x88, l_in_hw.device_address)
    write_64bit_address(control_r_ip, 0x94, u_in_hw.device_address)
    write_64bit_address(control_r_ip, 0xa0, q_in_hw.device_address)
    write_64bit_address(control_r_ip, 0xac, rho_in_hw.device_address)

    write_64bit_address(control_r_ip, 0xb8, x_out_hw.device_address)
    write_64bit_address(control_r_ip, 0xc4, y_out_hw.device_address)

    print("Starting Hardware Accelerator...")
    x_out_hw[:] = 0.0
    y_out_hw[:] = 0.0
    hw_start = time.time()

    control_ip.write(0x00, 0x01)
    while (control_ip.read(0x00) & 0x02) == 0:
        pass

    hw_end = time.time()
    print(f"HW Execution Time: {(hw_end - hw_start) * 1000:.4f} ms")

    admm_iters = int(control_ip.read(0x58))
    pcg_iters = int(control_ip.read(0x68))
    r_prim_out = float(uint_to_float(control_ip.read(0x78)))
    r_dual_out = float(uint_to_float(control_ip.read(0x88)))
    status_out = int(control_ip.read(0x98))

    x_hw = np.array(x_out_hw[:num_cols], dtype=np.float32)
    Ax_hw = (A_sparse @ x_hw).astype(np.float32)
    viol_hw = max_box_violation(Ax_hw, l_in, u_in)
    obj_hw = objective_val(x_hw)

    print("\n--- Results ---")
    print(f"Kernel Status: {'Converged' if status_out == 1 else 'Max Iterations Reached'}")
    print(f"ADMM Iterations: {admm_iters}")
    print(f"Total PCG Iterations: {pcg_iters}")
    print(f"Primal Residual (Ax-z): {r_prim_out:.5e}")
    print(f"Dual Residual   (KKT) : {r_dual_out:.5e}")
    print(f"Max constraint violation: {viol_hw:.3e}")
    print(f"Objective value: {obj_hw:.6e}")

    # Optional: print first N entries of x
    MAX_PRINT = 16
    nprint = min(num_cols, MAX_PRINT)
    print("\n--- x_out (truncated) ---")
    for i in range(nprint):
        print(f"x[{i:3}]: {x_hw[i]: .6f}")
    if num_cols > nprint:
        print(f"... ({num_cols - nprint} more not shown) ...")

    return {
        "adaptive_rho": int(adaptive_rho),
        "status": status_out,
        "admm_iters": admm_iters,
        "pcg_iters": pcg_iters,
        "r_prim": r_prim_out,
        "r_dual": r_dual_out,
        "viol": viol_hw,
        "obj": obj_hw,
        "hw_ms": (hw_end - hw_start) * 1000.0,
    }

adaptive_rho_list = [0, 1] if RUN_BOTH_ADAPTIVE_RHO else [0]
results = []
for adaptive_rho in adaptive_rho_list:
    results.append(run_hw(adaptive_rho))

if RUN_BOTH_ADAPTIVE_RHO and len(results) == 2:
    off = results[0]
    on = results[1]
    print("\n=== Summary (same dataset) ===")
    print(f"ADMM iters: off={off['admm_iters']} | on={on['admm_iters']}")
    print(f"Primal residual: off={off['r_prim']:.3e} | on={on['r_prim']:.3e}")
    print(f"Dual residual  : off={off['r_dual']:.3e} | on={on['r_dual']:.3e}")
    print(f"Max viol       : off={off['viol']:.3e} | on={on['viol']:.3e}")
    print(f"Obj value      : off={off['obj']:.6e} | on={on['obj']:.6e}")
    print(f"HW time (ms)   : off={off['hw_ms']:.3f} | on={on['hw_ms']:.3f}")

# Clean up memory
for buf in [A_row_words, A_val_words, AT_row_words, AT_val_words, P_row_words, P_val_words, A_col_ptr_hw, AT_col_ptr_hw, P_col_ptr_hw, P_diag_hw, l_in_hw, u_in_hw, q_in_hw, rho_in_hw, x_out_hw, y_out_hw]:
    buf.freebuffer()

Loading overlay from /home/xilinx/admm/admm.bit...
Overlay loaded!
Manually mapping Control   : 0xa0000000
Manually mapping Control_R : 0xa0010000

Running ADMM accelerator testbench...
Problem Size: 15x15
A_nnz: 120 (avg nnz/col: 8.00)
Data matrices generated successfully.
Matrices successfully dumped to: /home/xilinx/admm/baremetal/data
Allocating and Packing Hardware CMA Buffers (Uncached)...

=== HW Run (adaptive_rho=0) ===
Configuring Registers...
Starting Hardware Accelerator...
HW Execution Time: 0.6716 ms

--- Simulation Results ---
Status: Converged
ADMM Iterations: 10
Total PCG Iterations: 39
Primal Residual: 3.69228e-04
Dual Residual: 8.69170e-04

Mean Absolute Error from x_true: 1.69009e-02
--- Full x_out vs Expected ---
x[ 0]:       0.78853 | Expected:       0.78588
x[ 1]:      -0.98155 | Expected:      -0.85544
x[ 2]:      -1.09743 | Expected:      -1.09259
x[ 3]:       0.20772 | Expected:       0.20526
x[ 4]:       0.87914 | Expected:       0.87788
x[ 5]:      -0.30899 | Expected:      -0.30757
x[ 6]:       1.92965 | Expected:       1.92306
x[ 7]:       0.74201 | Expected:       0.73932
x[ 8]:      -0.07284 | Expected:      -0.07627
x[ 9]:      -0.43403 | Expected:      -0.43153
x[10]:      -0.61865 | Expected:      -0.62729
x[11]:       0.92262 | Expected:       0.91620
x[12]:      -0.25732 | Expected:      -0.24571
x[13]:      -1.83053 | Expected:      -1.76129
x[14]:      -0.41149 | Expected:      -0.40782

>>> FAILED: Did not converge to the expected target. <<<

=== HW Run (adaptive_rho=1) ===
Configuring Registers...
Starting Hardware Accelerator...
HW Execution Time: 0.6642 ms

--- Simulation Results ---
Status: Converged
ADMM Iterations: 10
Total PCG Iterations: 39
Primal Residual: 3.69228e-04
Dual Residual: 8.69170e-04

Mean Absolute Error from x_true: 1.69009e-02
--- Full x_out vs Expected ---
x[ 0]:       0.78853 | Expected:       0.78588
x[ 1]:      -0.98155 | Expected:      -0.85544
x[ 2]:      -1.09743 | Expected:      -1.09259
x[ 3]:       0.20772 | Expected:       0.20526
x[ 4]:       0.87914 | Expected:       0.87788
x[ 5]:      -0.30899 | Expected:      -0.30757
x[ 6]:       1.92965 | Expected:       1.92306
x[ 7]:       0.74201 | Expected:       0.73932
x[ 8]:      -0.07284 | Expected:      -0.07627
x[ 9]:      -0.43403 | Expected:      -0.43153
x[10]:      -0.61865 | Expected:      -0.62729
x[11]:       0.92262 | Expected:       0.91620
x[12]:      -0.25732 | Expected:      -0.24571
x[13]:      -1.83053 | Expected:      -1.76129
x[14]:      -0.41149 | Expected:      -0.40782

>>> FAILED: Did not converge to the expected target. <<<

=== Summary (same dataset) ===
ADMM iters: off=10 | on=10
Primal residual: off=3.692e-04 | on=3.692e-04
Dual residual  : off=8.692e-04 | on=8.692e-04
HW time (ms)   : off=0.672 | on=0.664
